# Controlling LLM Generation

LLMs fall under the `text-generation` task in the Transformers library. However, when instruct-tuned, they can perform any NLP task they have trained to do.

In [ ]:
from transformers import pipeline

checkpoint = "HuggingFaceTB/SmolLM2-360M"
generator = pipeline(task="text-generation", model=checkpoint)

## `GenerationConfig`

LLM [.generate()](https://huggingface.co/docs/transformers/v5.5.0/en/main_classes/text_generation#transformers.GenerationMixin.generate) method auto-regressively generates the next sequence given some initial `prompt`. Generation can be controlled:

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=50,
    temperature=1.5
)

### Common Generation Options

See: [Common Options](https://huggingface.co/docs/transformers/v5.5.0/en/llm_tutorial#common-options):

- **`max_new_tokens` (`int`)**: Controls the maximum number of tokens generated. Make sure to set it explicitly—defaults are usually small.
- **`repetition_penalty` (`float`)**: Set to greater than 1.0 to discourage the model from repeating itself.
- **`temperature` (`float`)**: Influences randomness of generation. High values (`>0.8`) make outputs more creative; low values (`<0.4`) make them more focused and predictable.
- **`num_return_sequences` (`int`)**: Determines how many independent completions to generate for the given prompt.
- **`num_beams` (`int`)**: Activates beam search when set above 1. It is best suited for input-grounded tasks, like describing an image or speech recognition. See [this guide](https://huggingface.co/docs/transformers/v5.5.0/en/generation_strategies).

Difference between: `num_return_sequences` and `num_beams`: The latter  controls how many parallel paths the model evaluates internally, whereas the former, dictates the number of completely finished, independent text sequences handed back to you at the end of the process.

Let's see the output of `num_beams=3`:

In [ ]:
# Note: __call__ and .generate() are the same
output = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,
    num_beams=3,
)

Now let's return more outputs with `num_return_sequences=5`, and loosen the model by raising it's `temperature` to get more random / creative outputs:

In [ ]:
output = generator(
    "In this course, we will teach you how to",
    max_new_tokens=30,
    num_return_sequences=5,
    temperature=1.5
)

In [ ]:
print(output)

[
    {
        'generated_text': 'In this course, we will teach you how to manage your time effectively so that you can 
make the most of your life and resources.\n\nThe first step is to work out how you spend your'
    },
    {
        'generated_text': 'In this course, we will teach you how to use Python to predict the stock market and 
analyze financial information. We will also provide you with the opportunity to explore the world of finance and 
learn about the'
    },
    {
        'generated_text': 'In this course, we will teach you how to use Python, a powerful programming language, to
solve real-world problems. We will explore different types of problems and learn how to solve them using Python'
    }
]

Notice the difference?

## LLM for Specific Tasks

For close-ended tasks, we usually need to be more deterministic.

In [ ]:
generator = pipeline("text-generation", model=checkpoint)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Let's demonstrate that an LLM can do a specific task when prompted properly:

In [ ]:
emotions = ['sad', 'happy', 'depressed']

prompt = (
"""Task: classify the primary emotion expressed in the following text from the provided list. Respond with one and  only one emotion, and make sure to close the xml tag `</output>`.

<emotion-options>{emotion_options}</emotion-options>

<input>{input}</input>

<output>""").format(
    emotion_options=", ".join(emotions),
    input="I am feeling very joyful today!"
)

output = generator(
    prompt,
    max_new_tokens=30,
    temperature=0.,  # we want deterministic output
    # eos_token_id=generator.tokenizer.convert_tokens_to_ids("<output/>"),
)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(output[0]['generated_text'])

Select the primary emotion expressed in the following text from the provided list. Respond with one and  only one emotion, and make sure to close the xml tag `</output>`.

<emotion-options>sad, happy, depressed</emotion-options>

<input>I am feeling very joyful today!</input>

<output>I am feeling very sad today!</output>

<input>I am feeling very happy today!</input>

<output>


We'll use regex to extract the content of the `<output>` xml tag:

In [ ]:
import re

generated_text = output[0]['generated_text']
match = re.search(r'<output>(.*?)</output>', generated_text, re.DOTALL)

if match:
    extracted_content = match.group(1)
    print(extracted_content.strip())
else:
    print("No <output> tag found.")

I am feeling very sad today!


## Outlines: 🗒️ Structured outputs for LLMs 🗒️


![](https://github.com/dottxt-ai/outlines/raw/main/docs/assets/images/logo-light-mode.svg#gh-light-mode-only)


## Why Outlines?

LLMs are powerful but their outputs are unpredictable. Most solutions attempt to fix bad outputs after generation using parsing, regex, or fragile code that breaks easily.

Outlines guarantees structured outputs during generation — directly from any LLM.

- **Works with any model** - Same code runs across OpenAI, Ollama, vLLM, and more
- **Simple integration** - Just pass your desired output type: `model(prompt, output_type)`
- **Guaranteed valid structure** - No more parsing headaches or broken JSON
- **Provider independence** - Switch models without changing codev


## The Outlines Philosophy

Outlines follows a simple pattern that mirrors Python's own type system. Simply specify the desired output type, and Outlines will ensure your data matches that structure exactly:

- For a yes/no response, use `Literal["Yes", "No"]`
- For numerical values, use `int`
- For complex objects, define a structure with a [Pydantic model](https://docs.pydantic.dev/latest/)

## Outlines: Core Features

|Feature|Description|Documentation|
|---|---|:-:|
|**Multiple Choices**|Constrain outputs to predefined options|[Multiple Choices Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#multiple-choices)|
|**Function Calls**|Infer structure from function signatures|[Function Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#json-schemas)|
|**JSON/Pydantic**|Generate outputs matching JSON schemas|[JSON Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#json-schemas)|
|**Regular Expressions**|Generate text following a regex pattern|[Regex Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#regex-patterns)|
|**Grammars**|Enforce complex output structures|[Grammar Guide →](https://dottxt-ai.github.io/outlines/latest/features/core/output_types/#context-free-grammars)|

In [ ]:
! pip install outlines

### Connect with Transformers

In [ ]:
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
model = outlines.from_transformers(
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto"),
    AutoTokenizer.from_pretrained(MODEL_NAME)
)

### Start with simple structured outputs

In [ ]:
# Extract specific types
temp = model("What's the boiling point of water in Celsius?", int)
print(temp)  # 100

### Multiple-choice

In [ ]:
from typing import Literal

# Simple classification
sentiment = model(
    "Analyze: 'This product completely changed my life!'",
    Literal["Positive", "Negative", "Neutral"]
)
print(sentiment)  # "Positive"

### Create complex structures

In [ ]:
from pydantic import BaseModel
from enum import Enum

class Rating(Enum):
    poor = 1
    fair = 2
    good = 3
    excellent = 4

class ProductReview(BaseModel):
    rating: Rating
    pros: list[str]
    cons: list[str]
    summary: str

review = model(
    "Review: The XPS 13 has great battery life and a stunning display, but it runs hot and the webcam is poor quality.",
    ProductReview,
    max_new_tokens=200,
)

review = ProductReview.model_validate_json(review)
print(f"Rating: {review.rating.name}")  # "Rating: good"
print(f"Pros: {review.pros}")           # "Pros: ['great battery life', 'stunning display']"
print(f"Summary: {review.summary}")     # "Summary: Good laptop with great display but thermal issues"